# Helper functions


In [ ]:
import numpy as np 
import pandas as pd 

import os
import sys
import requests
import shutil
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import timm
import kagglehub


print(" Installing dependencies and cloning repositories...")
!pip install -q timm yacs iopath kagglehub
!pip install -q git+https://github.com/fra31/auto-attack.git
!pip install -U safetensors


os.chdir('/content')
if os.path.exists('/content/SAR'): shutil.rmtree('/content/SAR')
if os.path.exists('/content/test-time-adaptation'): shutil.rmtree('/content/test-time-adaptation')

!git clone https://github.com/mr-eggplant/SAR.git /content/SAR
!git clone --depth 1 https://github.com/mariodoebler/test-time-adaptation.git /content/test-time-adaptation


sys.path.append('/content/SAR')
sys.path.insert(0, '/content/test-time-adaptation/classification')


import sar
def stable_softmax_entropy(x):
    return -(x.softmax(1) * x.log_softmax(1)).sum(1)
sar.softmax_entropy = stable_softmax_entropy


import types
import robustbench
try:
    import robustbench.loaders as rb_loaders
except ImportError:
    rb_loaders = types.ModuleType('loaders')
    robustbench.loaders = rb_loaders
    sys.modules['robustbench.loaders'] = rb_loaders

class DummyDataset(torch.utils.data.Dataset):
    def __init__(self, *args, **kwargs): pass
    def __len__(self): return 0
rb_loaders.CustomCifarDataset = DummyDataset
rb_loaders.CustomImageFolder = datasets.ImageFolder

import torch, gc

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print(" GPU Memory Cleared")


clear_gpu()


import os
import torch
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from tqdm import tqdm
import requests


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using device: {device}")


data_path = '/kaggle/input/datasets/sampadramkumar/gaussian-noise/kaggle/working/gaussian_noise_5'

# VIT


In [ ]:
import os
import torch
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from tqdm import tqdm
import requests


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using device: {device}")


data_path = '/kaggle/input/datasets/sampadramkumar/gaussian-noise/kaggle/working/gaussian_noise_5'

import logging
logging.basicConfig(level=logging.INFO)

print(" Loading VIT and native transforms...")

print(" Loading ViT (this may download weights, takes ~1–2 min)...")

model = timm.create_model(
    'vit_base_patch16_224',
    pretrained=True
).to(device).float()

print(" Model ready!")
model.eval()

config = resolve_data_config({}, model=model)
timm_transform = create_transform(**config)


print(f" Loading data from: {data_path}")
dataset = datasets.ImageFolder(root=data_path, transform=timm_transform)


print(" Remapping WordNet IDs to ImageNet classes...")
url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"
mapping = requests.get(url).json()
wnid_to_idx = {v[0]: int(k) for k, v in mapping.items()}


new_samples = [(p, wnid_to_idx[os.path.basename(os.path.dirname(p))])
               for p, _ in dataset.samples if os.path.basename(os.path.dirname(p)) in wnid_to_idx]
dataset.samples = new_samples
dataset.targets = [s[1] for s in new_samples]

print(f" Dataset ready. Total images: {len(dataset)} (Should be exactly 50,000)")


dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)


correct = 0
total = 0

print(" Running Base Evaluation...")
with torch.no_grad():
    for images, labels in tqdm(dataloader, desc="Evaluating", leave=True):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        pred = outputs.argmax(1)

        correct += (pred == labels).sum().item()
        total += labels.size(0)


final_accuracy = 100 * correct / total
print(f"\n Final Base Accuracy: {final_accuracy:.2f}%")

# ViT + SAR


In [ ]:
clear_gpu() 


In [ ]:



import os
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.datasets as datasets
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from tqdm import tqdm

from sar import SAR
from sam import SAM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Using device: {device}")






print(" Initializing Clean SAR Pipeline (ViT-Base)...")
sar_base_clean_vit = timm.create_model('vit_base_patch16_224', pretrained=True).to(device).float()


config = resolve_data_config({}, model=sar_base_clean_vit)
timm_transform = create_transform(**config)

print(f" Loading data from: {data_path}")
dataset = datasets.ImageFolder(root=data_path, transform=timm_transform)

print(" Remapping WordNet IDs to ImageNet classes...")
url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"
mapping = requests.get(url).json()
wnid_to_idx = {v[0]: int(k) for k, v in mapping.items()}

new_samples = [(p, wnid_to_idx[os.path.basename(os.path.dirname(p))])
               for p, _ in dataset.samples if os.path.basename(os.path.dirname(p)) in wnid_to_idx]
dataset.samples = new_samples
dataset.targets = [s[1] for s in new_samples]

dataloader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)




def configure_model_and_params_vit(model):

    model.train()
    model.requires_grad_(False)
    params = []

    for name, m in model.named_modules():
        
        if any(block_name in name for block_name in ['blocks.9', 'blocks.10', 'blocks.11']):
            continue

        if isinstance(m, nn.LayerNorm):
            m.requires_grad_(True)

            
            for np_name, p in m.named_parameters():
                if np_name in ['weight', 'bias']:
                    params.append(p)

    return params


params_to_adapt = configure_model_and_params_vit(sar_base_clean_vit)




margin_e0 = 0.4 * torch.log(torch.tensor(1000.0)).to(device)


optimizer_clean_vit = SAM(params_to_adapt, optim.SGD, lr=0.001, momentum=0.9, rho=0.05)
sar_model_clean_vit = SAR(sar_base_clean_vit, optimizer=optimizer_clean_vit, steps=1, episodic=False, margin_e0=margin_e0)

correct, total = 0, 0
threshold = margin_e0.item()

pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc="SAR ViT (Clean)")
for i, (images, labels) in pbar:
    images, labels = images.to(device).float(), labels.to(device)

    
    outputs = sar_model_clean_vit(images)

    
    probs = F.softmax(outputs, dim=1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-6), dim=1)
    pass_rate = (entropy < threshold).float().mean().item()

    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
    acc = 100 * correct / total

    pbar.set_postfix({
        "Acc": f"{acc:.2f}%",
        "Pass%": f"{pass_rate*100:.1f}"
    })

print(f"\n Final Clean SAR (ViT-Base) Accuracy: {acc:.2f}%")